importing Required modules

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn import preprocessing as per
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt 
from sklearn_pandas import DataFrameMapper
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.decomposition import KernelPCA
from lifelines.utils import concordance_index
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,median_absolute_error


import deepsurvk
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNetCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

G:\ana\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
G:\ana\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported con

In [2]:
#READING CSV FILE
df1 = pd.read_csv("data_bcr_clinical_data_patient.csv",na_values='?')
#EXCEPT CLINICAL DATA OTHERS HAVE PATIENT IDs WITH -01, SO ADD -01 AT THE END
df1.at[4,"Patient Identifier"]
def ankfunc(s):
    return s+"-01"
for i in range(4,532):
    df1.at[i,"Patient Identifier"]=ankfunc(df1.at[i,"Patient Identifier"])

#DROP ROWS AND COLUMNS
df1.drop([0,1,2,3] , inplace=True)
df1.set_index("Patient Identifier", inplace=True)

    
df1.replace('unknown',np.nan , inplace=True)
df1.replace('[Not Available]',np.nan , inplace=True)

df1.fillna(df1.mean(), inplace=True)
df1

C:\Users\PRODEE~1\AppData\Local\Temp/ipykernel_7464/2649448865.py:18: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df1.fillna(df1.mean(), inplace=True)


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NaN,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NaN,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [3]:
df1.fillna(method='ffill', inplace=True)
df1.fillna(method='bfill', inplace=True)
df1

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,1:Recurred/Progressed,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NOT HISPANIC OR LATINO,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NOT HISPANIC OR LATINO,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [4]:
df1['Lymph node neck dissection indicator'].replace(['[Not Available]','NO','YES'],['00','1','2'],inplace=True)

df1['Overall Survival Status'].replace(['0:LIVING','1:DECEASED'],['0','1'],inplace=True)
df1['Patient Primary Tumor Site'].replace(['[Not Available]','Buccal Mucosa','Larynx','Oral Cavity','Floor of mouth','Tonsil','Hypopharynx','Alveolar Ridge','Hard Palate','Oropharynx','Lip','Base of tongue','Oral Tongue'],['00','1','2','3','4','5','6','7','8','9','10','11','12'],inplace=True)
df1['Sex'].replace(['[Not Available]','Male','Female'],['00','1','2'],inplace=True)
df1['Race Category'].replace(['[Not Available]','WHITE','BLACK OR AFRICAN AMERICAN','ASIAN','AMERICAN INDIAN OR ALASKA NATIVE'],['00','1','2','3','4'],inplace=True)
df1['Ethnicity Category'].replace(['[Not Available]','NOT HISPANIC OR LATINO','HISPANIC OR LATINO'],['00','1','2'],inplace=True)
df1['Prior Cancer Diagnosis Occurence'].replace(['[Not Available]','No','Yes','Yes, History of Synchronous/Bilateral Malignancy','Yes, History of Prior Malignancy'],['00','1','2','3','4'],inplace=True)
df1['Neoadjuvant Therapy Type Administered Prior To Resection Text'].replace(['[Not Available]','No','Yes'],['00','1','2'],inplace=True)
df1['Vital Status'].replace(['[Not Available]','Dead','Alive'],['00','1','2'],inplace=True)
df1['American Joint Committee on Cancer Publication Version Type'].replace(['[Not Available]','6th','7th','5th','4th'],['00','1','2','3','4'],inplace=True)
df1['American Joint Committee on Cancer Tumor Stage Code'].replace(['[Not Available]','T0','T1','T2','T3','T4','T4a','T4b','TX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Disease Free Status'].replace(['0:DiseaseFree','1:Recurred/Progressed','[Not Available]'],['0','1','00'],inplace=True)
df1['Neoplasm Histologic Grade'].replace(['[Not Available]','G1','G2','G3','GX','G4'],['00','1','2','3','4','5'],inplace=True)
df1['Alcohol History Documented'].replace(['[Not Available]','No','Yes','NO','YES'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','N0','N1','N2','N2a','N2b','N2c','N3','NX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm Disease Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','Discrepancy','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','00','1','2','3','4','5','6'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage'].replace(['[Not Available]','M1','M1','MX','M0'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage'].replace(['[Not Available]','N0','N1','N2a','N2b','N2c','N3','NX','N2'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage'].replace(['[Not Available]','T1','T2','T3','T4a','T4b','TX','T4'],['00','1','2','3','4','5','6','7'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Group Stage'].replace(['[Not Available]','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','1','2','3','4','5','6'],inplace=True)

df1.head()

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,12,1,2,1,1,1,2013,2,2,2,...,66,4,3,4,4,0,0,3.35,0,3.35
TCGA-BA-4074-01,12,1,1,1,1,1,2003,2,1,1,...,69,4,5,3,4,0,1,15.18,1,13.01
TCGA-BA-4075-01,12,1,2,1,2,2,2004,2,1,1,...,49,4,2,4,4,0,1,9.3,1,7.75
TCGA-BA-4076-01,2,1,1,1,1,1,2003,2,1,1,...,39,4,5,3,4,0,1,13.63,1,9.4
TCGA-BA-4077-01,11,2,1,1,2,2,2003,2,1,1,...,45,4,6,5,5,0,1,37.25,1,9.4


In [5]:
#STORING REDUCED DATA TO CSV
cl = pd.DataFrame(df1)
cl.to_csv("CLINICALpreprocessed.csv")
print("Data exported to csv file")

Data exported to csv file


In [6]:
#READING CSV FILE
df =pd.read_csv("data_methylation_hm450.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df2 = df.T
df2.to_csv("Methylation.csv")
df2.head()

Hugo_Symbol,TSEN34,MUSTN1,C3orf16,CKLF,SFRS7,FAM180B,PTPRF,C6orf168,LOC728024,DSTYK,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
TCGA-4P-AA8J-01,0.160658,0.856690,0.689981,0.063268,0.088333,0.738956,0.689032,0.485472,0.847822,0.018144,...,0.021239,0.095778,0.061835,0.043785,0.055341,0.066497,0.080675,0.041548,0.054317,0.098915
TCGA-BA-4074-01,0.172720,0.888797,0.448310,0.095680,0.054274,0.506644,0.842746,0.188788,0.916802,0.018413,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
TCGA-BA-4075-01,0.091838,0.876359,0.336352,0.079018,0.062922,0.475571,0.783786,0.221566,0.792347,0.021204,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
TCGA-BA-4076-01,0.127324,0.911893,0.757925,0.095460,0.073372,0.834641,0.718938,0.791580,0.898727,0.014496,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
TCGA-BA-4077-01,0.132946,0.893790,0.556940,0.074819,0.080927,0.773512,0.392731,0.283078,0.857577,0.018026,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227


In [7]:
# Merge datasets based on the patient identifier
merged_df = pd.merge(cl, df2, left_index=True, right_index=True, how="inner")
df2=merged_df


In [8]:

c2 = pd.DataFrame(df2)
c2.to_csv("Methylationpreprocessed.csv")

In [9]:

# Extract target variable (survival time) from clinical data
y = c2['Overall Survival (Months)']
c2 = c2.drop('Overall Survival (Months)', axis=1)


In [10]:
y.drop(y.index[-1], inplace=True)
dm=c2.iloc[:,:]
#print(d)
dm = dm.reset_index()
M=dm.iloc[1:,1:]
M


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
1,12,1,1,1,1,1,2003,2,1,1,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
2,12,1,2,1,2,2,2004,2,1,1,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
3,2,1,1,1,1,1,2003,2,1,1,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
4,11,2,1,1,2,2,2003,2,1,1,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227
5,2,1,1,1,1,1,2003,2,1,1,...,0.036186,0.065871,0.042553,0.043962,0.036412,0.131708,0.062926,0.043616,0.044998,0.057528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,4,2,1,1,1,1,2009,2,1,1,...,0.019317,0.045418,0.032703,0.032664,0.035815,0.044621,0.080887,0.037113,0.033082,0.048041
524,6,2,1,1,3,1,2011,2,1,2,...,0.030936,0.068638,0.042534,0.043967,0.047700,0.063594,0.094378,0.045045,0.049718,0.084455
525,12,1,1,2,1,1,2013,2,2,2,...,0.024656,0.044608,0.030440,0.047566,0.064576,0.062233,0.051034,0.034986,0.053662,0.051514
526,4,1,1,1,1,1,2012,2,1,2,...,0.026150,0.078658,0.047960,0.048287,0.050811,0.059785,0.077223,0.051496,0.048270,0.059510


In [11]:
#standardize the data
scaler = StandardScaler()
X1 = scaler.fit_transform(M[:])
#PCA 
# fit pca on data
pca = KernelPCA(n_components=387, kernel='rbf', gamma=15, random_state=42)



Z1=pca.fit_transform(X1)


In [12]:
n1 = pd.DataFrame(Z1)
print(n1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
n1["Hugo"]=hugo
n1

(527, 387)


,0,1,2,3,4,5,6,7,8,9,...,378,379,380,381,382,383,384,385,386,Hugo
0,0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,-0.000000,...,0.000000,0.000000,0.000000,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,0.000000,TCGA-BA-4074-01
1,0.005605,-0.003433,0.005142,-0.000799,-0.001513,0.000877,0.005441,0.001079,0.000826,0.000293,...,-0.002350,0.005868,-0.001524,-0.005722,0.002600,0.000666,0.001420,-0.003414,-0.004808,TCGA-BA-4075-01
2,-0.005681,0.002868,-0.003785,0.004949,-0.000200,0.000200,-0.004952,-0.000546,-0.004383,0.000290,...,0.004185,-0.007004,0.000904,0.009206,-0.002877,-0.001647,-0.001650,0.000980,0.006334,TCGA-BA-4076-01
3,0.006515,-0.003150,0.026141,0.001982,-0.009382,0.010217,0.007635,0.002097,0.001245,-0.003594,...,-0.013584,0.011210,0.005273,-0.018306,0.005133,0.001945,-0.002852,-0.006998,-0.007474,TCGA-BA-4077-01
4,0.015074,-0.007591,-0.013720,0.005625,0.003203,-0.000773,0.010117,-0.001004,-0.010724,-0.000637,...,0.006014,-0.008942,-0.010264,0.009174,-0.002718,-0.008759,0.009345,-0.001945,0.004633,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-0.017790,-0.003057,0.004165,-0.012530,0.004091,-0.002339,-0.043462,0.004578,0.001590,0.002600,...,-0.007691,0.017178,-0.000849,0.001862,0.030684,-0.002472,-0.012302,0.040397,-0.001106,TCGA-UF-A7JT-01
523,0.044984,0.003257,-0.003503,-0.038452,-0.020266,0.023612,0.002487,-0.016687,0.002004,-0.018031,...,0.010578,0.005565,-0.048429,0.007082,-0.028023,-0.031503,-0.056768,-0.132839,-0.007944,TCGA-UF-A7JV-01
524,-0.031110,-0.002047,0.036223,-0.055254,0.011253,-0.016165,0.019110,-0.037122,0.004079,0.024652,...,0.005239,-0.004015,-0.006991,0.042429,0.032078,-0.021527,-0.027555,-0.035898,0.040516,TCGA-UP-A6WW-01
525,-0.034673,-0.009487,0.005757,-0.002995,0.003930,0.002560,0.022372,-0.003509,-0.003844,-0.000818,...,-0.000735,-0.000054,-0.022661,0.003662,0.004742,0.023490,-0.013200,0.016957,0.001974,TCGA-WA-A7GZ-01


In [13]:

lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z1, y)
n_components = Z1.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(38).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zl1 = np.column_stack(columns)
nl1 = pd.DataFrame(Zl1)
print(nl1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nl1["Hugo"]=hugo
nl1

Number of selected features:  38
Selected features:  Int64Index([ 63, 174, 291, 332, 381,  64,  17, 186, 178, 114, 113, 311, 192,
             44, 105,   1, 326, 129, 286, 184, 368, 102,   2,   3,   4,   5,
              6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  18],
           dtype='int64')
(527, 38)


,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,Hugo
0,-0.000000,0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,0.000000,0.000000,-0.000000,...,-0.000000,-0.000000,0.000000,-0.000000,-0.227019,-0.104243,0.000000,-0.000000,0.000000,TCGA-BA-4074-01
1,0.006094,0.000843,-0.003979,-0.009128,-0.005722,-0.003325,0.001980,0.002824,-0.000496,-0.000136,...,0.000293,0.001980,-0.002839,0.004634,0.105534,0.010208,-0.000287,0.001609,-0.004258,TCGA-BA-4075-01
2,-0.007074,0.000768,0.005718,0.008574,0.009206,0.002964,-0.005202,-0.002828,-0.003523,0.002080,...,0.000290,-0.002977,0.003648,-0.004451,0.185373,0.068146,0.002362,-0.004221,0.006174,TCGA-BA-4076-01
3,0.016238,0.006615,0.001649,-0.023330,-0.018306,-0.006539,0.001959,0.000940,-0.003276,0.000665,...,-0.003594,0.009655,-0.016976,0.002626,0.122314,0.221671,0.003183,0.006295,-0.015252,TCGA-BA-4077-01
4,-0.017911,-0.002523,-0.026580,-0.004907,0.009174,0.003464,-0.005673,0.006837,-0.005389,0.007794,...,-0.000637,-0.007857,0.018866,0.007564,-0.045775,0.116391,-0.005597,0.007153,0.015139,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.016557,0.004061,-0.004019,0.159219,0.001862,-0.004108,-0.000450,0.008052,-0.024891,0.012199,...,0.002600,0.005864,-0.000451,-0.000675,0.023062,0.006853,-0.003554,0.014620,0.004564,TCGA-UF-A7JT-01
523,-0.008326,0.024249,-0.000653,0.157565,0.007082,-0.000812,-0.012725,0.008949,0.012972,-0.049396,...,-0.018031,0.007109,-0.008002,-0.013793,0.011737,0.016821,-0.010778,-0.012680,-0.004301,TCGA-UF-A7JV-01
524,0.018586,-0.008318,0.024681,0.031266,0.042429,-0.018811,-0.005767,-0.028823,0.004738,0.017650,...,0.024652,0.006587,0.075064,-0.004663,0.055305,0.002000,0.077326,-0.031480,0.006359,TCGA-UP-A6WW-01
525,0.002481,0.002687,0.002414,-0.077581,0.003662,-0.001112,-0.001362,-0.001704,-0.008333,0.059852,...,-0.000818,-0.010076,-0.004653,-0.001905,0.027315,0.000738,-0.011297,0.016459,0.000013,TCGA-WA-A7GZ-01


In [14]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z1, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z1.shape[1])])
selected_features = coef.abs().nlargest(38).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Ze1 = np.column_stack(columns)
ne1 = pd.DataFrame(Ze1)
ne1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
ne1["Hugo"]=hugo
ne1


Number of selected features:  38
Selected features:  [63, 174, 291, 332, 381, 64, 17, 186, 178, 114, 113, 311, 192, 44, 105, 1, 326, 129, 286, 184, 368, 102, 165, 156, 160, 282, 365, 250, 33, 166, 40, 149, 214, 362, 22, 95, 277, 138]


,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,Hugo
0,-0.000000,0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,0.000000,0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.070917,0.000000,-0.000000,0.000000,TCGA-BA-4074-01
1,0.006094,0.000843,-0.003979,-0.009128,-0.005722,-0.003325,0.001980,0.002824,-0.000496,-0.000136,...,-0.005964,-0.005509,0.003657,-0.000138,0.002196,0.262646,-0.003390,0.003900,0.002515,TCGA-BA-4075-01
2,-0.007074,0.000768,0.005718,0.008574,0.009206,0.002964,-0.005202,-0.002828,-0.003523,0.002080,...,-0.000692,0.008607,-0.003786,0.002170,0.000760,-0.286993,0.001983,-0.001559,-0.003531,TCGA-BA-4076-01
3,0.016238,0.006615,0.001649,-0.023330,-0.018306,-0.006539,0.001959,0.000940,-0.003276,0.000665,...,-0.021470,-0.019416,0.014073,-0.002873,0.009781,-0.163987,-0.007269,0.004321,0.001832,TCGA-BA-4077-01
4,-0.017911,-0.002523,-0.026580,-0.004907,0.009174,0.003464,-0.005673,0.006837,-0.005389,0.007794,...,0.004145,0.008980,-0.014892,-0.002497,-0.006252,-0.036143,-0.001956,-0.000590,0.004457,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,0.016557,0.004061,-0.004019,0.159219,0.001862,-0.004108,-0.000450,0.008052,-0.024891,0.012199,...,0.010874,-0.008426,-0.033574,0.002187,0.018159,0.002160,0.021043,-0.002440,-0.002688,TCGA-UF-A7JT-01
523,-0.008326,0.024249,-0.000653,0.157565,0.007082,-0.000812,-0.012725,0.008949,0.012972,-0.049396,...,0.061405,0.022458,-0.033839,-0.013802,-0.003054,0.006512,0.027955,-0.001615,-0.027766,TCGA-UF-A7JV-01
524,0.018586,-0.008318,0.024681,0.031266,0.042429,-0.018811,-0.005767,-0.028823,0.004738,0.017650,...,0.047415,0.009911,-0.006895,-0.042829,-0.028073,-0.012156,0.064251,0.045390,0.024320,TCGA-UP-A6WW-01
525,0.002481,0.002687,0.002414,-0.077581,0.003662,-0.001112,-0.001362,-0.001704,-0.008333,0.059852,...,-0.011779,-0.006845,-0.012351,0.008437,0.010797,0.011161,0.015614,-0.010083,-0.001365,TCGA-WA-A7GZ-01


In [15]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z1, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z1.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(38)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zrf1 = np.column_stack(columns)
nrf1 = pd.DataFrame(Zrf1)
nrf1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nrf1["Hugo"]=hugo
nrf1

Number of selected features:  38
Selected features:  [291, 250, 380, 189, 129, 281, 192, 332, 184, 150, 63, 149, 178, 64, 209, 91, 33, 46, 311, 362, 42, 277, 173, 344, 247, 202, 282, 86, 54, 186, 138, 210, 313, 22, 262, 212, 288, 326]


,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,Hugo
0,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,-0.000000,...,0.000000,0.000000,-0.151450,0.000000,0.070917,0.000000,0.000000,-0.000000,-0.000000,TCGA-BA-4074-01
1,-0.003979,0.001483,-0.001524,-0.000140,0.001892,-0.002412,-0.001949,-0.009128,-0.008554,0.001463,...,0.002824,0.002515,0.222125,-0.007853,0.262646,-0.004608,-0.001809,-0.008190,-0.002503,TCGA-BA-4075-01
2,0.005718,-0.008900,0.000904,-0.000135,-0.008057,0.004745,0.002787,0.008574,0.015564,-0.000482,...,-0.002828,-0.003531,-0.023903,0.009562,-0.286993,0.003524,-0.003694,0.009535,0.002690,TCGA-BA-4076-01
3,0.001649,0.001783,0.005273,-0.001474,-0.007359,-0.003577,-0.011133,-0.023330,-0.009499,0.013676,...,0.000940,0.001832,0.248401,-0.011281,-0.163987,-0.013523,0.000141,-0.010297,0.006438,TCGA-BA-4077-01
4,-0.026580,-0.009144,-0.010264,0.001296,-0.009024,0.009806,0.012945,-0.004907,-0.005780,0.006554,...,0.006837,0.004457,-0.227931,-0.002590,-0.036143,0.011626,-0.016558,-0.012501,-0.021611,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-0.004019,0.238866,-0.000849,0.025714,0.136233,-0.019554,-0.020993,0.159219,0.019705,0.000592,...,0.008052,-0.002688,-0.011433,0.001227,0.002160,0.001253,0.035598,-0.000728,0.016584,TCGA-UF-A7JT-01
523,-0.000653,0.345731,-0.048429,0.033091,0.072343,0.011456,-0.006566,0.157565,-0.005953,-0.042943,...,0.008949,-0.027766,0.011122,-0.010609,0.006512,0.000714,0.001266,0.012486,0.053741,TCGA-UF-A7JV-01
524,0.024681,0.059742,-0.006991,-0.041915,0.083364,-0.043108,0.029610,0.031266,-0.044175,0.014325,...,-0.028823,0.024320,0.030114,0.051175,-0.012156,-0.006567,-0.011808,0.019164,-0.013032,TCGA-UP-A6WW-01
525,0.002414,-0.010665,-0.022661,0.017651,0.037451,0.020341,-0.010419,-0.077581,0.008713,-0.011355,...,-0.001704,-0.001365,0.007971,-0.005079,0.011161,0.003538,-0.001640,0.004305,0.007429,TCGA-WA-A7GZ-01


In [16]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=38)
rfe.fit(Z1, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zre1 = np.column_stack(columns)
nre1 = pd.DataFrame(Zre1)
nre1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nre1["Hugo"]=hugo
nre1

Selected Features: [1, 17, 22, 33, 40, 44, 63, 64, 95, 102, 105, 113, 114, 129, 138, 149, 156, 160, 165, 166, 174, 178, 184, 186, 192, 214, 250, 277, 282, 286, 291, 311, 326, 332, 362, 365, 368, 381]


,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,Hugo
0,-0.000000,-0.000000,0.070917,0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,...,0.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,-0.000000,TCGA-BA-4074-01
1,-0.003433,0.001980,0.262646,-0.001345,-0.005509,0.000383,0.006094,-0.003325,-0.003390,0.003281,...,0.001219,-0.003979,-0.000138,-0.002503,-0.009128,0.002196,0.006419,0.002422,-0.005722,TCGA-BA-4075-01
2,0.002868,-0.005202,-0.286993,-0.000350,0.008607,0.002037,-0.007074,0.002964,0.001983,-0.006254,...,-0.001051,0.005718,0.001914,0.002690,0.008574,0.000760,-0.008387,-0.000606,0.009206,TCGA-BA-4076-01
3,-0.003150,0.001959,-0.163987,0.000812,-0.019416,0.004205,0.016238,-0.006539,-0.007269,0.003743,...,0.006262,0.001649,-0.000194,0.006438,-0.023330,0.009781,0.002595,0.004812,-0.018306,TCGA-BA-4077-01
4,-0.007591,-0.005673,-0.036143,-0.014980,0.008980,-0.009086,-0.017911,0.003464,-0.001956,-0.011238,...,0.000029,-0.026580,0.011418,-0.021611,-0.004907,-0.006252,0.017469,0.005067,0.009174,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-0.003057,-0.000450,0.002160,-0.021275,-0.008426,-0.023115,0.016557,-0.004108,0.021043,0.048391,...,0.005051,-0.004019,-0.014951,0.016584,0.159219,0.018159,-0.004283,0.021183,0.001862,TCGA-UF-A7JT-01
523,0.003257,-0.012725,0.006512,0.024338,0.022458,0.018515,-0.008326,-0.000812,0.027955,0.027608,...,0.000500,-0.000653,0.003180,0.053741,0.157565,-0.003054,0.014077,0.048475,0.007082,TCGA-UF-A7JV-01
524,-0.002047,-0.005767,-0.012156,-0.009670,0.009911,0.001017,0.018586,-0.018811,0.064251,0.040033,...,-0.046047,0.024681,-0.001521,-0.013032,0.031266,-0.028073,0.049101,-0.069166,0.042429,TCGA-UP-A6WW-01
525,-0.009487,-0.001362,0.011161,0.003970,-0.006845,-0.004458,0.002481,-0.001112,0.015614,-0.001672,...,-0.000392,0.002414,-0.001397,0.007429,-0.077581,0.010797,-0.003341,0.001638,0.003662,TCGA-WA-A7GZ-01


In [17]:
#READING CSV FILE
df = pd.read_csv("data_RNA_Seq_v2_expression_median.csv")

#DROP ROWS AND COLUMNS
df=df.replace(0,np.nan)
df=df.dropna()
df=df.replace(np.nan,0)
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df4 = df.T
df4.to_csv("RNAb.csv")
df4.head()
merged_df = pd.merge(cl, df4, left_index=True, right_index=True, how="inner")
df4=merged_df

In [18]:
c4 = pd.DataFrame(df4)
c4.to_csv("RNApreprocessed.csv")
y = c4['Overall Survival (Months)']
c4 = c4.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
drn=c4.iloc[:,:]
#print(d)
drn = drn.reset_index()
Rn=drn.iloc[1:,1:]
Rn.head()


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,LOC154274,ZW10,ZWILCH,ZWINT,ZXDB,LOC100130182,ZYG11B,ZYX,FLJ10821,ZZZ3
1,12,1,1,1,1,1,2003,2,1,1,...,311.0030,283.6409,2132.4595,1193.1417,172.2656,380.3717,805.0624,2516.9279,258.5911,1088.3179
2,12,1,2,1,2,2,2004,2,1,1,...,225.1105,512.3945,761.0023,673.1877,172.0488,562.2404,487.7395,5930.0549,292.6437,980.3028
3,2,1,1,1,1,1,2003,2,1,1,...,157.9431,307.4905,480.0682,1032.6643,324.2818,1440.9025,722.5502,2674.5376,672.1763,998.5570
4,11,2,1,1,2,2,2003,2,1,1,...,137.6323,361.4052,1325.3128,1620.3080,210.7796,1423.0029,770.9336,8035.6112,763.2339,692.9740
5,2,1,1,1,1,1,2003,2,1,1,...,241.8520,414.1231,874.1257,1145.1112,372.9953,2634.2473,780.1345,3895.2406,1556.6477,1309.6223


Applying PCA dimensioality reduction technique

In [19]:
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X3 = scaler.fit_transform(Rn[:])
#fit pca on data
pca = KernelPCA(n_components=323, kernel='rbf', gamma=15, random_state=42)
Z3=pca.fit_transform(X3)



Applying RFE Feature selection methods

In [20]:
n3 = pd.DataFrame(Z3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
n3["Hugo"]=hugo
n3

,0,1,2,3,4,5,6,7,8,9,...,314,315,316,317,318,319,320,321,322,Hugo
0,-0.323783,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,TCGA-BA-4074-01
1,0.278697,0.003413,0.006377,0.002075,0.001031,-0.003621,0.003687,-0.005322,-0.001486,-0.000926,...,-0.005480,-0.002871,-0.007232,-0.008777,-0.005115,-0.001396,-0.002478,-0.006476,0.013193,TCGA-BA-4075-01
2,-0.202480,0.004514,-0.001217,-0.003257,0.011928,0.000643,0.019518,-0.007130,0.002804,0.012049,...,-0.008513,-0.001080,-0.001615,-0.009431,-0.004636,-0.012552,0.006963,-0.006652,0.008036,TCGA-BA-4076-01
3,-0.298378,-0.002476,-0.014162,-0.004064,-0.024749,-0.001602,-0.025865,0.033506,-0.001430,-0.014372,...,0.013169,0.018142,0.023344,0.012639,0.024415,0.019045,0.002949,0.022386,-0.033661,TCGA-BA-4077-01
4,-0.094394,-0.015955,-0.036776,-0.006820,-0.030439,0.034941,-0.018514,0.052876,0.002189,-0.005624,...,0.030013,0.019879,0.029305,0.032694,0.031324,0.018837,0.000183,0.025889,-0.066929,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-0.006394,0.007678,-0.002958,0.007915,0.001245,-0.011337,-0.006187,-0.009310,0.048930,-0.056900,...,-0.002697,-0.005507,0.005749,0.062936,-0.021452,0.035433,-0.035872,0.008979,0.005906,TCGA-UF-A7JT-01
515,-0.001950,-0.004968,-0.023767,-0.006892,0.002419,0.014588,0.000648,0.017563,-0.010429,0.036321,...,-0.012878,0.016117,0.015141,0.027195,0.059740,-0.004796,-0.007694,0.005260,-0.025531,TCGA-UF-A7JV-01
516,-0.014587,0.002502,-0.003836,-0.009952,0.014946,-0.014390,0.036769,-0.006197,0.015429,-0.079490,...,-0.029211,0.057032,-0.023254,0.026091,-0.004653,-0.086821,0.002546,-0.041126,-0.026261,TCGA-UP-A6WW-01
517,0.004925,0.007691,0.001073,-0.006884,0.004749,0.002090,-0.000537,-0.015689,0.000604,-0.018197,...,0.000671,0.014693,0.017987,0.007863,-0.030434,0.046552,-0.005777,-0.035766,-0.040151,TCGA-WA-A7GZ-01


In [21]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z3, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z3.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(32)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zrf3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf3 = pd.DataFrame(Zrf3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nrf3["Hugo"]=hugo
nrf3

Number of selected features:  32
Selected features:  [303, 115, 114, 39, 26, 188, 80, 250, 98, 135, 130, 132, 137, 16, 51, 305, 136, 147, 143, 261, 43, 253, 63, 183, 46, 22, 32, 163, 311, 74, 247, 186]


,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,Hugo
0,0.000000,0.000000,-0.000000,0.000000,0.000000,-0.000000,0.103361,0.000000,0.000000,0.000000,...,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,-0.000000,TCGA-BA-4074-01
1,-0.000694,-0.006524,0.003893,0.000639,-0.001562,-0.005328,-0.054383,0.006422,0.002071,0.000543,...,-0.004805,-0.001834,-0.005196,0.007849,-0.000253,-0.000824,-0.005221,0.003543,0.003147,TCGA-BA-4075-01
2,-0.012493,-0.006050,0.005410,-0.015485,-0.007132,-0.012249,0.297144,0.002726,-0.003627,0.001907,...,-0.000182,-0.007430,-0.010071,-0.009754,-0.015168,-0.000702,0.004869,-0.001084,0.000404,TCGA-BA-4076-01
3,-0.010974,0.037688,-0.017667,0.005544,0.026736,0.028990,-0.121005,-0.018254,-0.007652,-0.002685,...,0.007500,0.004414,0.011915,-0.009223,0.023329,0.018069,0.008544,0.001555,0.011791,TCGA-BA-4077-01
4,0.008969,0.037870,-0.027481,-0.010893,0.002727,0.031134,-0.243145,-0.041295,-0.029059,-0.006313,...,0.026678,0.006073,0.035184,-0.055601,-0.004200,0.010087,0.028325,-0.013198,-0.001475,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.015510,-0.012888,-0.020713,-0.008443,-0.003508,0.013320,-0.003573,0.010030,-0.005798,0.001196,...,0.027821,-0.000937,0.010250,-0.000096,0.005919,0.002174,-0.027200,-0.005199,-0.004407,TCGA-UF-A7JT-01
515,-0.029122,-0.043535,-0.011226,0.007093,-0.007489,-0.014025,-0.000317,-0.009461,-0.020642,-0.020463,...,-0.141402,0.008213,0.003990,-0.007594,0.025270,-0.023483,-0.003047,-0.020012,0.001521,TCGA-UF-A7JV-01
516,0.041717,0.014759,-0.015977,-0.013179,0.001771,0.026116,0.005352,0.005515,0.012997,-0.005217,...,0.018211,0.004905,-0.006600,-0.016474,-0.009294,0.020351,-0.040656,-0.006412,0.007233,TCGA-UP-A6WW-01
517,-0.025113,-0.022109,-0.022053,-0.015615,-0.009621,-0.024561,0.023035,0.049519,-0.005049,0.017048,...,-0.000581,-0.005918,-0.007991,-0.004283,0.009462,-0.011531,0.003211,-0.006501,-0.014603,TCGA-WA-A7GZ-01


In [22]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=32)
rfe.fit(Z3, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zre3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre3 = pd.DataFrame(Zre3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nre3["Hugo"]=hugo
nre3

Selected Features: [12, 16, 23, 26, 39, 54, 56, 74, 75, 80, 84, 98, 106, 114, 115, 125, 130, 135, 136, 137, 143, 147, 167, 186, 188, 197, 250, 253, 256, 269, 303, 317]


,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,Hugo
0,0.036179,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.103361,...,-0.000000,-0.000000,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,-0.000000,TCGA-BA-4074-01
1,0.077740,0.000148,0.000187,-0.001562,0.000639,-0.001964,-0.003130,-0.005221,0.007017,-0.054383,...,0.003147,-0.005328,-0.002397,0.006422,0.000124,0.002835,0.006077,-0.000694,-0.008777,TCGA-BA-4075-01
2,-0.045973,0.001877,-0.004223,-0.007132,-0.015485,-0.010614,0.001835,0.004869,-0.001997,0.297144,...,0.000404,-0.012249,-0.008633,0.002726,-0.000021,0.005949,-0.004228,-0.012493,-0.009431,TCGA-BA-4076-01
3,-0.002488,0.004302,0.020654,0.026736,0.005544,0.014900,0.005067,0.008544,-0.020666,-0.121005,...,0.011791,0.028990,0.024491,-0.018254,0.011186,-0.016564,-0.020553,-0.010974,0.012639,TCGA-BA-4077-01
4,0.000427,-0.010585,-0.007539,0.002727,-0.010893,0.013623,0.007408,0.028325,-0.055150,-0.243145,...,-0.001475,0.031134,0.012520,-0.041295,0.008323,-0.029745,-0.035902,0.008969,0.032694,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,0.003037,-0.007382,-0.008665,-0.003508,-0.008443,-0.004273,0.058812,-0.027200,0.000878,-0.003573,...,-0.004407,0.013320,-0.001729,0.010030,-0.010519,0.001587,-0.006763,0.015510,0.062936,TCGA-UF-A7JT-01
515,-0.020819,0.003708,0.037526,-0.007489,0.007093,-0.025375,-0.040413,-0.003047,-0.026604,-0.000317,...,0.001521,-0.014025,0.016729,-0.009461,0.017609,0.003690,0.002718,-0.029122,0.027195,TCGA-UF-A7JV-01
516,-0.035528,0.000175,0.006566,0.001771,-0.013179,-0.006749,-0.019683,-0.040656,-0.003563,0.005352,...,0.007233,0.026116,-0.001999,0.005515,0.011821,-0.021091,0.011834,0.041717,0.026091,TCGA-UP-A6WW-01
517,-0.005955,0.024222,0.000861,-0.009621,-0.015615,-0.030809,0.040829,0.003211,0.000560,0.023035,...,-0.014603,-0.024561,-0.006270,0.049519,0.003670,-0.000960,0.019603,-0.025113,0.007863,TCGA-WA-A7GZ-01


In [23]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z3, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z3.shape[1])])
selected_features = coef.abs().nlargest(32).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Ze3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne3 = pd.DataFrame(Ze3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
ne3["Hugo"]=hugo
ne3


Number of selected features:  32
Selected features:  [114, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,Hugo
0,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,TCGA-BA-4074-01
1,0.003893,0.003413,0.006377,0.002075,0.001031,-0.003621,0.003687,-0.005322,-0.001486,-0.000926,...,0.000187,0.003449,-0.007865,-0.001562,0.002028,0.006814,-0.002257,0.003147,-0.010567,TCGA-BA-4075-01
2,0.005410,0.004514,-0.001217,-0.003257,0.011928,0.000643,0.019518,-0.007130,0.002804,0.012049,...,-0.004223,0.009940,-0.007935,-0.007132,0.001473,-0.000534,-0.000998,-0.001565,-0.008087,TCGA-BA-4076-01
3,-0.017667,-0.002476,-0.014162,-0.004064,-0.024749,-0.001602,-0.025865,0.033506,-0.001430,-0.014372,...,0.020654,-0.020236,0.037122,0.026736,0.010864,-0.023886,0.008241,-0.013841,0.041478,TCGA-BA-4077-01
4,-0.027481,-0.015955,-0.036776,-0.006820,-0.030439,0.034941,-0.018514,0.052876,0.002189,-0.005624,...,-0.007539,-0.016610,0.052119,0.002727,-0.001380,-0.038346,0.012507,-0.013333,0.061020,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-0.020713,0.007678,-0.002958,0.007915,0.001245,-0.011337,-0.006187,-0.009310,0.048930,-0.056900,...,-0.008665,-0.000324,0.017350,-0.003508,-0.003857,0.018482,0.036784,-0.115245,0.003451,TCGA-UF-A7JT-01
515,-0.011226,-0.004968,-0.023767,-0.006892,0.002419,0.014588,0.000648,0.017563,-0.010429,0.036321,...,0.037526,0.010300,0.021718,-0.007489,-0.005412,-0.004709,0.149438,-0.039836,0.015228,TCGA-UF-A7JV-01
516,-0.015977,0.002502,-0.003836,-0.009952,0.014946,-0.014390,0.036769,-0.006197,0.015429,-0.079490,...,0.006566,0.003320,-0.006037,0.001771,-0.009637,-0.057536,-0.017774,0.009017,0.011663,TCGA-UP-A6WW-01
517,-0.022053,0.007691,0.001073,-0.006884,0.004749,0.002090,-0.000537,-0.015689,0.000604,-0.018197,...,0.000861,-0.012539,0.008368,-0.009621,-0.020510,-0.001223,-0.011280,-0.003444,0.009222,TCGA-WA-A7GZ-01


In [24]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z3, y)
n_components = Z3.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(32).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zl3 = np.column_stack(columns)
nl3 = pd.DataFrame(Zl3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nl3["Hugo"]=hugo
nl3

Number of selected features:  32
Selected features:  Int64Index([114,  26, 115, 303, 256, 130,  23, 136,  39,  98, 106,  54, 186,
            250, 135,  75,  16,  80, 167, 147,  12,   1,   2,   3,   4,   5,
              6,   7,   8,   9,  10,  11],
           dtype='int64')


,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,Hugo
0,-0.000000,0.000000,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.234460,TCGA-BA-4074-01
1,0.003893,-0.001562,-0.006524,-0.000694,0.002835,-0.006770,0.000187,0.001083,0.000639,0.002071,...,0.002075,0.001031,-0.003621,0.003687,-0.005322,-0.001486,-0.000926,-0.004566,-0.269180,TCGA-BA-4075-01
2,0.005410,-0.007132,-0.006050,-0.012493,0.005949,-0.011066,-0.004223,-0.003939,-0.015485,-0.003627,...,-0.003257,0.011928,0.000643,0.019518,-0.007130,0.002804,0.012049,0.003835,-0.133894,TCGA-BA-4076-01
3,-0.017667,0.026736,0.037688,-0.010974,-0.016564,0.029698,0.020654,-0.005984,0.005544,-0.007652,...,-0.004064,-0.024749,-0.001602,-0.025865,0.033506,-0.001430,-0.014372,0.012182,0.107750,TCGA-BA-4077-01
4,-0.027481,0.002727,0.037870,0.008969,-0.029745,0.029283,-0.007539,-0.007441,-0.010893,-0.029059,...,-0.006820,-0.030439,0.034941,-0.018514,0.052876,0.002189,-0.005624,0.024440,-0.081643,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-0.020713,-0.003508,-0.012888,0.015510,0.001587,-0.000524,-0.008665,-0.008897,-0.008443,-0.005798,...,0.007915,0.001245,-0.011337,-0.006187,-0.009310,0.048930,-0.056900,-0.071767,0.012385,TCGA-UF-A7JT-01
515,-0.011226,-0.007489,-0.043535,-0.029122,0.003690,0.007112,0.037526,-0.010458,0.007093,-0.020642,...,-0.006892,0.002419,0.014588,0.000648,0.017563,-0.010429,0.036321,-0.194996,-0.020792,TCGA-UF-A7JV-01
516,-0.015977,0.001771,0.014759,0.041717,-0.021091,0.005046,0.006566,0.007033,-0.013179,0.012997,...,-0.009952,0.014946,-0.014390,0.036769,-0.006197,0.015429,-0.079490,0.009513,0.039240,TCGA-UP-A6WW-01
517,-0.022053,-0.009621,-0.022109,-0.025113,-0.000960,0.003805,0.000861,-0.036163,-0.015615,-0.005049,...,-0.006884,0.004749,0.002090,-0.000537,-0.015689,0.000604,-0.018197,-0.034245,0.024259,TCGA-WA-A7GZ-01


In [ ]:
df = pd.read_csv("data_linear_CNA.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df5 = df.T
df5.to_csv("CNA.csv")
df5.head()
merged_df = pd.merge(cl, df5, left_index=True, right_index=True, how="inner")
df5=merged_df
c5 = pd.DataFrame(df5)
c5.to_csv("CNApreprocessed.csv")
y = c5['Overall Survival (Months)']
c5 = c5.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
dcn=c5.iloc[:,:]
#print(d)
dcn = dcn.reset_index()
Cn=dcn.iloc[1:,1:]
Cn.head()
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X4 = scaler.fit_transform(Cn[:])
#fit pca on data
pca = KernelPCA(n_components=161, kernel='rbf', gamma=15, random_state=42)

Z4=pca.fit_transform(X4)
Z4

n4 = pd.DataFrame(Z4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
n4["Hugo"]=hugo
n4

In [ ]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z4, y)
n_components = Z4.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(16).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zl4 = np.column_stack(columns)
nl4 = pd.DataFrame(Zl4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nl4["Hugo"]=hugo
nl4

In [ ]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z4, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z4.shape[1])])
selected_features = coef.abs().nlargest(16).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Ze4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne4 = pd.DataFrame(Ze4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    ne4["Hugo"]=hugo
except Exception:
    pass


In [ ]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=16)
rfe.fit(Z4, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zre4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre4 = pd.DataFrame(Zre4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    nre4["Hugo"]=hugo
except Exception:
    pass

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z4, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z4.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(16)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zrf4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf4 = pd.DataFrame(Zrf4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nrf4["Hugo"]=hugo
nrf4

Merging dataset

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(n1['Hugo'][i]==n3['Hugo'][j]):
            z.append(n1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==n4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,n1,on='Hugo')
mdf=pd.merge(mdf,n3,on='Hugo')
mdf=pd.merge(mdf,n4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




In [ ]:
import deepsurvk

dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)


In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nl1['Hugo'][i]==nl3['Hugo'][j]):
            z.append(nl1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nl4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nl1,on='Hugo')
mdf=pd.merge(mdf,nl3,on='Hugo')
mdf=pd.merge(mdf,nl4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 10
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv)")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)



In [ ]:
result = np.zeros((5, 4, 5))

result[0][0][0]=c_index_test
result[0][0][1]=mse
result[0][0][2]=rmse
result[0][0][3]=mae
result[0][0][4]=mdae

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][0][0]=c_index
result[1][0][1]=mse
result[1][0][2]=r2
result[1][0][3]=mae
result[1][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][0][0]=c_index
result[2][0][1]=mse
result[2][0][2]=r2
result[2][0][3]=mae
result[2][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][0][0]=c_index
result[3][0][1]=mse
result[3][0][2]=r2
result[3][0][3]=mae
result[3][0][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][0][0]=c_index
result[4][0][1]=mse
result[4][0][2]=r2
result[4][0][3]=mae
result[4][0][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(ne1['Hugo'][i]==ne3['Hugo'][j]):
            z.append(ne1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==ne4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,ne1,on='Hugo')
mdf=pd.merge(mdf,ne3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][1][0]=c_index_test
result[0][1][1]=mse
result[0][1][2]=rmse
result[0][1][3]=mae
result[0][1][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][1][0]=c_index
result[1][1][1]=mse
result[1][1][2]=r2
result[1][1][3]=mae
result[1][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][1][0]=c_index
result[2][1][1]=mse
result[2][1][2]=r2
result[2][1][3]=mae
result[2][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][1][0]=c_index
result[3][1][1]=mse
result[3][1][2]=r2
result[3][1][3]=mae
result[3][1][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][1][0]=c_index
result[4][1][1]=mse
result[4][1][2]=r2
result[4][1][3]=mae
result[4][1][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nre1['Hugo'][i]==nre3['Hugo'][j]):
            z.append(nre1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nre4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nre1,on='Hugo')
mdf=pd.merge(mdf,nre3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][2][0]=c_index_test
result[0][2][1]=mse
result[0][2][2]=rmse
result[0][2][3]=mae
result[0][2][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][2][0]=c_index
result[1][2][1]=mse
result[1][2][2]=r2
result[1][2][3]=mae
result[1][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][2][0]=c_index
result[2][2][1]=mse
result[2][2][2]=r2
result[2][2][3]=mae
result[2][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][2][0]=c_index
result[3][2][1]=mse
result[3][2][2]=r2
result[3][2][3]=mae
result[3][2][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][2][0]=c_index
result[4][2][1]=mse
result[4][2][2]=r2
result[4][2][3]=mae
result[4][2][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nrf1['Hugo'][i]==nrf3['Hugo'][j]):
            z.append(nrf1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nrf4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nrf1,on='Hugo')
mdf=pd.merge(mdf,nrf3,on='Hugo')
mdf=pd.merge(mdf,nrf4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)

# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][3][0]=c_index_test
result[0][3][1]=mse
result[0][3][2]=rmse
result[0][3][3]=mae
result[0][3][4]=mdae



In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][3][0]=c_index
result[1][3][1]=mse
result[1][3][2]=r2
result[1][3][3]=mae
result[1][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][3][0]=c_index
result[2][3][1]=mse
result[2][3][2]=r2
result[2][3][3]=mae
result[2][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][3][0]=c_index
result[3][3][1]=mse
result[3][3][2]=r2
result[3][3][3]=mae
result[3][3][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][3][0]=c_index
result[4][3][1]=mse
result[4][3][2]=r2
result[4][3][3]=mae
result[4][3][4]=mdae

In [ ]:
!pip install pandas openpyxl
data_0_0 = result[0, 0:4, 0]
data_1_0 = result[3, 0:4, 0]
data_2_0 = result[2, 0:4, 0]
data_3_0 = result[1, 0:4, 0]
data_4_0 = result[4, 0:4, 0]
combined_data_0 = np.concatenate((data_0_0, data_1_0, data_2_0, data_3_0, data_4_0))
data_0_1 = result[0, 0:4, 1]
data_1_1 = result[3, 0:4, 1]
data_2_1 = result[2, 0:4, 1]
data_3_1 = result[1, 0:4, 1]
data_4_1 = result[4, 0:4, 1]
combined_data_1 = np.concatenate((data_0_1, data_1_1, data_2_1, data_3_1, data_4_1))
data_0_2 = result[0, 0:4, 2]
data_1_2 = result[3, 0:4, 2]
data_2_2 = result[2, 0:4, 2]
data_3_2 = result[1, 0:4, 2]
data_4_2 = result[4, 0:4, 2]
combined_data_2 = np.concatenate((data_0_2, data_1_2, data_2_2, data_3_2, data_4_2))
data_0_3 = result[0, 0:4, 3]
data_1_3 = result[3, 0:4, 3]
data_2_3 = result[2, 0:4, 3]
data_3_3 = result[1, 0:4, 3]
data_4_3 = result[4, 0:4, 3]
combined_data_3 = np.concatenate((data_0_3, data_1_3, data_2_3, data_3_3, data_4_3))
data_0_4 = result[0, 0:4, 4]
data_1_4 = result[3, 0:4, 4]
data_2_4 = result[2, 0:4, 4]
data_3_4 = result[1, 0:4, 4]
data_4_4 = result[4, 0:4, 4]
combined_data_4 = np.concatenate((data_0_4, data_1_4, data_2_4, data_3_4, data_4_4))
# Create a DataFrame
df = pd.DataFrame({
    'Combined_Data_0': combined_data_0,
    'Combined_Data_1': combined_data_1,
    'Combined_Data_2': combined_data_2,
    'Combined_Data_3': combined_data_3,
    'Combined_Data_4': combined_data_4
})

# Save the DataFrame to an Excel file
df.to_excel('combined_data.xlsx', index=False)

print("Data has been saved to combined_data.xlsx")